# VAE for Text Style Transfer — Starter Notebook

## Architecture Overview

```
Input Text
    │
    ▼
┌─────────────────────┐
│  LLaMA 3.1 Tokenizer │  ← Pretrained BPE tokenizer (128K vocab)
└─────────┬───────────┘
          │ token_ids: (batch, seq_len)
          ▼
┌─────────────────────┐
│  LLaMA 3.1 Embedding │  ← Pretrained embeddings (128K × 4096)
│  (frozen or finetune)│
└─────────┬───────────┘
          │ embeddings: (batch, seq_len, 4096)
          ▼
┌─────────────────────┐
│    VAE Encoder       │  ← TODO: your encoder network
│  (→ mu, log_var)     │
└─────────┬───────────┘
          │ z: (batch, latent_dim)
          ▼
┌─────────────────────┐
│    VAE Decoder       │  ← TODO: your decoder network
│  (+ style condition) │
└─────────┬───────────┘
          │
          ▼
    Output Text
```

This notebook implements the **top two blocks** (Tokenizer + Embedding) using LLaMA 3.1 8B.

In [5]:
# Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, AutoConfig
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
import json

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

PyTorch version: 2.10.0+cu128
CUDA available: True
Using device: cuda


## 1. Load the LLaMA 3.1 Tokenizer

In [2]:
# Login to hf
from huggingface_hub import login
login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
MODEL_NAME = "meta-llama/Llama-3.1-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# LLaMA doesn't have a pad token by default — we need one for batching
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Pad token: '{tokenizer.pad_token}' (id={tokenizer.pad_token_id})")
print(f"EOS token: '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")
print(f"BOS token: '{tokenizer.bos_token}' (id={tokenizer.bos_token_id})")

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Vocab size: 128000
Pad token: '<|end_of_text|>' (id=128001)
EOS token: '<|end_of_text|>' (id=128001)
BOS token: '<|begin_of_text|>' (id=128000)


In [4]:
# Quick test — see how the tokenizer handles style-transfer-relevant text
examples = [
    "The food was absolutely terrible and I hated every bite.",   # negative
    "The food was absolutely wonderful and I loved every bite.",  # positive
    "The meal was acceptable.",                                   # neutral
]

for text in examples:
    tokens = tokenizer(text, return_tensors="pt")
    decoded_tokens = tokenizer.convert_ids_to_tokens(tokens["input_ids"][0])
    print(f"\nText: {text}")
    print(f"  Token IDs shape: {tokens['input_ids'].shape}")
    print(f"  Tokens: {decoded_tokens}")


Text: The food was absolutely terrible and I hated every bite.
  Token IDs shape: torch.Size([1, 12])
  Tokens: ['<|begin_of_text|>', 'The', 'Ġfood', 'Ġwas', 'Ġabsolutely', 'Ġterrible', 'Ġand', 'ĠI', 'Ġhated', 'Ġevery', 'Ġbite', '.']

Text: The food was absolutely wonderful and I loved every bite.
  Token IDs shape: torch.Size([1, 12])
  Tokens: ['<|begin_of_text|>', 'The', 'Ġfood', 'Ġwas', 'Ġabsolutely', 'Ġwonderful', 'Ġand', 'ĠI', 'Ġloved', 'Ġevery', 'Ġbite', '.']

Text: The meal was acceptable.
  Token IDs shape: torch.Size([1, 6])
  Tokens: ['<|begin_of_text|>', 'The', 'Ġmeal', 'Ġwas', 'Ġacceptable', '.']


**Note**:

The Ġ character is how LLaMA's BPE tokenizer represents a leading space. So Ġfood just means " food". It's an implementation detail of byte-level BPE — every token that doesn't start a sentence gets the space baked into the token itself rather than having a separate space token.


## 2. Extract the Pretrained Embedding Layer

- We pull **only** the embedding layer from LLaMA 3.1 8B — not the full model.  
This saves a ton of VRAM (~32GB model → ~0.5GB for just the embedding table).
- Sources:
  - https://ai.stackexchange.com/questions/45054/why-do-llm-tokenizers-use-a-special-symbol-for-space-such-as-%C4%A0-in-bpe-or-in-sp
  - https://github.com/dleemiller/WordLlama/blob/main/tutorials/extract_token_embeddings.md

In [ ]:
# Sample code from HF website
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_id)
inputs = tokenizer("Hello, how are you?", return_tensors="pt").to(model.device)

# Get only the embedded vectors
with torch.no_grad():
    embedded_vectors = model.get_input_embeddings()(inputs['input_ids'])

print(embedded_vectors.shape)
# Result: [batch, sequence_length, 4096]

In [ ]:
# ---------- Option A: Extract from the full model (if you have the VRAM / disk) ----------
# This loads the full model, copies the embedding weights, then deletes the model.

def load_pretrained_embeddings_from_model(model_name: str) -> nn.Embedding:
    """Load the full model, extract embedding weights, discard the rest."""
    print("Loading full model to extract embeddings (this may take a minute)...")
    model = AutoModel.from_pretrained(
        model_name,
        torch_dtype=torch.float16,   # save memory
        device_map="cpu",            # keep on CPU for extraction
    )
    embed_weight = model.embed_tokens.weight.clone().float()  # (vocab_size, hidden_dim)
    del model  # free memory
    torch.cuda.empty_cache()

    vocab_size, embed_dim = embed_weight.shape
    print(f"Extracted embedding: vocab_size={vocab_size}, embed_dim={embed_dim}")

    embedding_layer = nn.Embedding(vocab_size, embed_dim)
    embedding_layer.weight = nn.Parameter(embed_weight)
    return embedding_layer

In [ ]:
# ---------- Option B (RECOMMENDED): Load only the embedding weights via config ----------
# Uses HF's model sharding to avoid loading the full 8B model.

def load_pretrained_embeddings_lightweight(model_name: str) -> nn.Embedding:
    """
    Load only the embedding layer without loading the full model.
    Uses safetensors index to find and load just the embedding shard.
    """

    # Download the safetensors index to find which shard has the embeddings
    index_path = hf_hub_download(model_name, "model.safetensors.index.json")
    with open(index_path) as f:
        index = json.load(f)

    # Find the shard containing 'model.embed_tokens.weight'
    embed_key = "model.embed_tokens.weight"
    shard_file = index["weight_map"][embed_key]
    print(f"Embedding is in shard: {shard_file}")

    # Download only that shard
    shard_path = hf_hub_download(model_name, shard_file)
    shard = load_file(shard_path)
    embed_weight = shard[embed_key].float()  # (vocab_size, 4096)

    vocab_size, embed_dim = embed_weight.shape
    print(f"Extracted embedding: vocab_size={vocab_size}, embed_dim={embed_dim}")

    embedding_layer = nn.Embedding(vocab_size, embed_dim)
    embedding_layer.weight = nn.Parameter(embed_weight)
    return embedding_layer

In [9]:
# Option B simplified
from safetensors import safe_open
from huggingface_hub import hf_hub_download
import json
import torch
from torch import nn

def load_pretrained_embeddings_surgical(model_name: str) -> nn.Embedding:
    # 1. Get the index
    index_path = hf_hub_download(model_name, "model.safetensors.index.json")
    with open(index_path) as f:
        index = json.load(f)

    # 2. Locate shard
    embed_key = "model.embed_tokens.weight"
    shard_file = index["weight_map"][embed_key]
    shard_path = hf_hub_download(model_name, shard_file)

    # 3. USE SAFE_OPEN (This is the secret sauce)
    # This maps the file to memory without loading the whole thing.
    with safe_open(shard_path, framework="pt", device="cpu") as f:
        # Pull ONLY the specific tensor you want
        embed_weight = f.get_tensor(embed_key)

    vocab_size, embed_dim = embed_weight.shape

    # Keep it in its native dtype (bfloat16) to save 50% RAM,
    # unless you specifically need float32.
    embedding_layer = nn.Embedding(vocab_size, embed_dim, _weight=embed_weight)

    return embedding_layer

In [10]:
# ===== Pick one and run it =====
# Uncomment the one that works for your setup:

# embedding_layer = load_pretrained_embeddings_from_model(MODEL_NAME)   # Option A
# embedding_layer = load_pretrained_embeddings_lightweight(MODEL_NAME)  # Option B (recommended)
embedding_layer = load_pretrained_embeddings_surgical(MODEL_NAME)

embedding_layer = embedding_layer.to(DEVICE)
print(f"\nEmbedding layer on: {next(embedding_layer.parameters()).device}")
print(f"Embedding weight shape: {embedding_layer.weight.shape}")
print(f"Memory: {embedding_layer.weight.nelement() * 4 / 1e9:.2f} GB (float32)")


Embedding layer on: cuda:0
Embedding weight shape: torch.Size([128256, 4096])
Memory: 2.10 GB (float32)


## 3. Tokenize + Embed Pipeline

This is the function we'll call at the top of your VAE forward pass.

In [11]:
MAX_SEQ_LEN = 128  # adjust for your dataset


def tokenize_and_embed(
    texts: list[str],
    tokenizer,
    embedding_layer: nn.Embedding,
    max_length: int = MAX_SEQ_LEN,
) -> dict:
    """
    Full pipeline: raw text → token IDs → dense embeddings.

    Returns:
        dict with:
            - 'input_ids':       (batch, seq_len)          LongTensor
            - 'attention_mask':  (batch, seq_len)          LongTensor
            - 'embeddings':      (batch, seq_len, 4096)    FloatTensor
    """
    # Step 1: Tokenize
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    input_ids = encoded["input_ids"].to(DEVICE)           # (batch, seq_len)
    attention_mask = encoded["attention_mask"].to(DEVICE)  # (batch, seq_len)

    # Step 2: Embed
    with torch.no_grad():  # remove this if you want to finetune embeddings
        embeddings = embedding_layer(input_ids)  # (batch, seq_len, embed_dim)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "embeddings": embeddings,
    }

In [12]:
# Test the full pipeline
test_texts = [
    "This movie was absolutely fantastic, a true masterpiece!",
    "This movie was absolutely dreadful, a complete waste of time.",
]

result = tokenize_and_embed(test_texts, tokenizer, embedding_layer)

print("Pipeline output shapes:")
for k, v in result.items():
    print(f"  {k}: {v.shape} ({v.dtype})")


Pipeline output shapes:
  input_ids: torch.Size([2, 13]) (torch.int64)
  attention_mask: torch.Size([2, 13]) (torch.int64)
  embeddings: torch.Size([2, 13, 4096]) (torch.bfloat16)


## 4. VAE Stub — Your Starting Point

Below is a skeleton for the VAE. The encoder/decoder are **placeholders** — replace later with actual architecture (ie Transformer encoder blocks).

In [ ]:
class TextStyleVAE(nn.Module):
    """
    VAE for text style transfer.

    Architecture:
        Pretrained Tokenizer + Embedding (this notebook)
        → Encoder → mu, log_var
        → Reparameterize → z
        → Decoder (conditioned on style label) → reconstructed text
    """

    def __init__(
        self,
        embedding_layer: nn.Embedding,
        latent_dim: int = 256,
        num_styles: int = 2,           # e.g. positive / negative
        freeze_embeddings: bool = True,
    ):
        super().__init__()

        self.embedding = embedding_layer
        embed_dim = embedding_layer.weight.shape[1]  # 4096 for LLaMA 3.1

        if freeze_embeddings:
            for param in self.embedding.parameters():
                param.requires_grad = False

        # ──── Encoder (TODO: replace with your real encoder) ────
        # Example: simple mean-pooling + linear projection
        self.encoder_proj = nn.Linear(embed_dim, latent_dim * 2)  # outputs mu and log_var

        # ──── Style embedding ────
        self.style_embedding = nn.Embedding(num_styles, latent_dim)

        # ──── Decoder (TODO: replace with your real decoder) ────
        # Example: linear projection back to vocab logits
        self.decoder_proj = nn.Linear(latent_dim * 2, embed_dim)  # z + style → embed space
        self.output_head = nn.Linear(embed_dim, embedding_layer.weight.shape[0])  # → vocab logits

    def encode(self, embeddings: torch.Tensor, attention_mask: torch.Tensor):
        """
        Encode embeddings to latent space.

        TODO: Replace this with a proper encoder (e.g., bidirectional GRU,
              Transformer encoder, etc.)
        """
        # Placeholder: mean pooling over sequence
        mask = attention_mask.unsqueeze(-1).float()  # (batch, seq_len, 1)
        pooled = (embeddings * mask).sum(dim=1) / mask.sum(dim=1)  # (batch, embed_dim)

        h = self.encoder_proj(pooled)               # (batch, latent_dim * 2)
        mu, log_var = h.chunk(2, dim=-1)             # each (batch, latent_dim)
        return mu, log_var

    def reparameterize(self, mu: torch.Tensor, log_var: torch.Tensor) -> torch.Tensor:
        """Reparameterization trick: z = mu + std * epsilon."""
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z: torch.Tensor, style_ids: torch.Tensor, seq_len: int):
        """
        Decode latent vector to token logits.

        TODO: Replace with an autoregressive decoder (e.g., GRU with teacher
              forcing, Transformer decoder, etc.)
        """
        style = self.style_embedding(style_ids)      # (batch, latent_dim)
        combined = torch.cat([z, style], dim=-1)      # (batch, latent_dim * 2)

        # Placeholder: project back and repeat for each timestep
        h = self.decoder_proj(combined)               # (batch, embed_dim)
        h = h.unsqueeze(1).expand(-1, seq_len, -1)    # (batch, seq_len, embed_dim)
        logits = self.output_head(h)                  # (batch, seq_len, vocab_size)
        return logits

    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        style_ids: torch.Tensor,
    ):
        # Step 1: Embed
        embeddings = self.embedding(input_ids)        # (batch, seq_len, embed_dim)

        # Step 2: Encode
        mu, log_var = self.encode(embeddings, attention_mask)

        # Step 3: Reparameterize
        z = self.reparameterize(mu, log_var)

        # Step 4: Decode
        logits = self.decode(z, style_ids, seq_len=input_ids.shape[1])

        return logits, mu, log_var

In [ ]:
# ──── VAE Loss Function ────

def vae_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    mu: torch.Tensor,
    log_var: torch.Tensor,
    attention_mask: torch.Tensor,
    kl_weight: float = 0.1,    # beta for beta-VAE; tune this!
) -> dict:
    """
    Standard VAE loss = Reconstruction + KL Divergence.

    Args:
        logits:         (batch, seq_len, vocab_size)
        targets:        (batch, seq_len) — the input_ids (reconstruction target)
        mu, log_var:    (batch, latent_dim)
        attention_mask: (batch, seq_len) — ignore padding in recon loss
        kl_weight:      scaling factor for KL term (beta-VAE)
    """
    # Reconstruction loss (cross-entropy per token, masked)
    recon_loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        targets.view(-1),
        reduction="none",
    ).view(targets.shape)  # (batch, seq_len)

    recon_loss = (recon_loss * attention_mask).sum() / attention_mask.sum()

    # KL divergence: -0.5 * sum(1 + log_var - mu^2 - exp(log_var))
    kl_loss = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp(), dim=-1).mean()

    total_loss = recon_loss + kl_weight * kl_loss

    return {
        "loss": total_loss,
        "recon_loss": recon_loss.item(),
        "kl_loss": kl_loss.item(),
    }

## 5. Smoke Test — Forward Pass

In [ ]:
# Instantiate the VAE
model = TextStyleVAE(
    embedding_layer=embedding_layer,
    latent_dim=256,
    num_styles=2,
    freeze_embeddings=True,
).to(DEVICE)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Frozen params:    {total_params - trainable_params:,} (embeddings)")

In [ ]:
# Run a forward pass with dummy data
texts = [
    "The restaurant had terrible service and cold food.",
    "What a wonderful dining experience, truly outstanding!",
]
style_labels = torch.tensor([0, 1]).to(DEVICE)  # 0=negative, 1=positive

# Tokenize
encoded = tokenizer(texts, padding=True, truncation=True, max_length=MAX_SEQ_LEN, return_tensors="pt")
input_ids = encoded["input_ids"].to(DEVICE)
attention_mask = encoded["attention_mask"].to(DEVICE)

# Forward
model.train()
logits, mu, log_var = model(input_ids, attention_mask, style_labels)

print(f"Logits shape:   {logits.shape}   (batch, seq_len, vocab_size)")
print(f"Mu shape:       {mu.shape}       (batch, latent_dim)")
print(f"Log_var shape:  {log_var.shape}  (batch, latent_dim)")

# Compute loss
losses = vae_loss(logits, input_ids, mu, log_var, attention_mask, kl_weight=0.1)
print(f"\nTotal loss:  {losses['loss']:.4f}")
print(f"Recon loss:  {losses['recon_loss']:.4f}")
print(f"KL loss:     {losses['kl_loss']:.4f}")

print("\n✅ Forward pass works! You're ready to build out the encoder/decoder.")

## 6. Next Steps / TODOs

The tokenizer + embedding front-end is done. Here's what to tackle next:

1. **Encoder**: Replace the mean-pooling placeholder with a proper sequence encoder  
   - Bidirectional GRU/LSTM  
   - Small Transformer encoder (2-4 layers)  
   - Consider adding a projection layer (4096 → 512) before the encoder to reduce compute  

2. **Decoder**: Replace the stub with an autoregressive decoder  
   - GRU with teacher forcing during training  
   - Transformer decoder with causal masking  
   - Style conditioning via concatenation, addition, or cross-attention  

3. **Training loop**: Add optimizer, KL annealing schedule, gradient clipping  

4. **Dataset**: Load a style-transfer dataset (e.g., Yelp sentiment, Shakespeare ↔ Modern English)  

5. **Evaluation**: BLEU, style accuracy (classifier), content preservation metrics